# ResNet-50 Image Classification model

In [1]:
import torch 
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
from easydict import EasyDict as edict
import random

/home/ma-user/anaconda3/envs/PyTorch-1.8/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Define constant Dictionary

In [2]:
cfg = edict({
    'data_path':'./flower_photos/flower_photos_train',
    'test_path':'./flower_photos/flower_photos_test',
    'data_size':3616,
    'HEIGHT':128,
    'WIDTH':128,

    '_R_MEAN':123.68,
    '_G_MEAN':116.78,
    '_B_MEAN':103.94,
    '_R_STD':1,
    '_G_STD':1,
    '_B_STD':1,
    '_RESIZE_SIDE_MIN':256,
    '_RESIZE_SIDE_MAX':512,

    'batch_size':1,
    'num_class':5,
    'epoch_size':3,
    'num_workers':1,

    'device': torch.device("cuda" if torch.cuda.is_available() else "cpu"),

    'prefix':'resnet50.pth'
})





### define the data and process data

In [3]:
#Use transform .Compose
transform_train=transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[cfg._R_MEAN, cfg._G_MEAN, cfg._B_MEAN], std=[cfg._R_STD, cfg._G_STD, cfg._B_STD
                                                                            
                            ]),
    transforms.Resize(cfg._RESIZE_SIDE_MIN),
    transforms.CenterCrop((cfg.HEIGHT, cfg.WIDTH)),
])

transform_test=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

### Load data

In [4]:
train=torchvision.datasets.ImageFolder(cfg.data_path, transform=transform_train)
trainloader=torch.utils.data.DataLoader(train, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers)


### Define the network structure

In [5]:
#Define the basic block, which is mainly used for resnet-18
class BasicBlock(nn.Module):
    expansion=1
    def __init__(self , in_channels, out_channels, stride=1):
        super(BasicBlock,self).__init__()
        
        self.residual_function=nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3 , stride=stride, padding=1, bias=False),
            #Second Convolutional
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels*BasicBlock.expansion, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels*BasicBlock.expansion)
            
        )
        self.shortcut=nn.Sequential()
        if stride !=1 or in_channels !=BasicBlock.expansion*out_channels:
            self.shortcut=nn.Sequential(
            nn.Conv2d(in_channels, out_channels*BasicBlock.expansion, kernel_size=1, stride=stride, bias=False),
            nn.BatchNorm2d(out_channels*BasicBlock.expansion))
            
    def forwad(self ,x):
        return nn.ReLU(inplace=True)(self.residual_function(x)+ self.shortcut(x))
 #Define Bottlneck class
class BottleNeck(nn.Module):
    expansion=4
    def __init__(self , in_channels, out_channels, stride=1):
        super(BottleNeck, self).__init__()
        
        self.residual_function=nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=1 , bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels, stride=stride, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels*BottleNeck.expansion, kernel_size=1, bias=False),
        nn.BatchNorm2d(out_channels*BottleNeck.expansion))
        
        
        self.shortcut=nn.Sequential()
        
        if stride !=1 or in_channels !=BottleNeck.expansion*out_channels:
            self.shortcut=nn.Sequential(
            nn.Conv2d(in_channels, out_channels*BottleNeck.expansion, kernel_size=1, stride=stride, bias=False),
            nn.BatchNorm2d(out_channels*BottleNeck.expansion))
            
    def forward(self ,x):
        return nn.ReLU(inplace=True)(self.residual_function(x)+ self.shortcut(x))
    
    
# Define the ResNet network
class ResNet(nn.Module):
    def __init__(self , block, num_block, num_classes=100):
        super(ResNet, self).__init__()
        
        self.in_channels=64
        
        # The first convolution layer
        self.conv1=nn.Sequential(
        nn.Conv2d(3,64, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(64), 
        nn.ReLU(inplace=True))
        # the number of layer varies depending on hte ResNet Version
        self.conv2_x=self._make_layer(block,64,num_block[0], 1)
        self.conv3_x=self._make_layer(block,128,num_block[1], 2)
        self.conv4_x=self._make_layer(block,256,num_block[2], 2)
        self.conv5_x=self._make_layer(block,512,num_block[3], 2)
        
        # Global average pooling layer
        self.avg_pool=nn.AdaptiveAvgPool2d((1,1))
        
        # Fully-Connected layer Set the number of output class as required
        self.fc=nn.Linear(512*block.expansion, num_classes)
        
    def _make_layer(self, block , out_channels, num_blocks, stride):
        strides=[stride]+[1]*(num_blocks-1)
        layers=[]
        for stride in strides:
            layers.append(block(self.in_channels, out_channels, stride)) # Add a block
            self.in_channels=out_channels*block.expansion  #update number of channels
        return nn.Sequential(*layers) #Return layer sequence
    
    def forward(self, x):
        # Forward Propagation
        output=self.conv1(x)
        output=self.conv2_x(output)
        output=self.conv3_x(output)
        output=self.conv4_x(output)
        output=self.conv5_x(output)
        
        output=self.avg_pool(output) # Global average pooling
        output=output.view(output.size(0), -1) #Flatten into one dimension vector
        output = self.fc(output) #Fully-connected layer
        
        return output
        
        
    
    
            

### Define ResNet Diffent Version

In [6]:
def resnet18():
    """Return the ResNet-18 Model"""
    return ResNet(BasicBlock,[2,2,2,2] )

 
def resnet34():
    """Return the ResNet-34 Model"""
    return ResNet(BasicBlock,[3,4,6,3])

def resnet50():
    """Return the ResNet-50 Model"""
    return ResNet(BottleNeck,[3,4,6,3],num_classes=cfg.num_class)
# change then number of classes


def resnet101():
    """Return the ResNet-101 Model"""
    return ResNet(BottleNeck, [3,4,23,3])

def resnet152():
    """Return the ResNet-152 Model"""
    return ResNet(BottleNeck,[3,8,36,3])



## Resnet-50 is used in this experiment

### Define loss function and optimizer , and initialize the network

In [7]:
import os
net=resnet50().to(cfg.device)
if os.path.isfile(cfg.prefix):
    net.load_state_dict(torch.load(cfg.prefix))
    
criterion=nn.CrossEntropyLoss()    
optimizer=optim.SGD(net.parameters(), lr=0.1, momentum=0.9, weight_decay=0.0001)
scheduler=optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.1, patience=5)

### Define Training Process

In [ ]:
for epoch in range(cfg.epoch_size):
    losses = []
    running_loss = 0.0

    for i, inp in enumerate(trainloader):
        inputs, labels = inp
        inputs, labels = inputs.to(cfg.device), labels.to(cfg.device)

        optimizer.zero_grad()

        outputs = net(inputs)
        loss = criterion(outputs, labels)

        losses.append(loss.item())

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        if i % 200 == 0 and i > 0:
            print(f'Loss [{epoch+1},{i}] (epoch, minibatch):',
                  running_loss / 200)
            running_loss = 0.0

    avg_loss = sum(losses) / len(losses)
    scheduler.step(avg_loss)

print('Training Done')


### Save the model

In [ ]:
print('Model is reached')
torch.save(net.state_dict(), cfg.prefix)


### Test the model

In [ ]:
test=torchvision.datasets.ImageFolder(cf.test_path, transform=transform_test)
transloader=torch.utils=torch.utils.data.DataLoader(test, batch_size, shuffle=True, num_workers=cfg.num_workers)

correct=0
total=0
with torch.no_grad():
    for data in testloader:
        images, labels=data
        images, labels=images.to(cfg.device) , labels.to(cfg.device)
        outputs=net(images)
        
        _,predicted=torch.max(outputs.data,1)
        total +=labels.size(0)
        correct+=(predicted==labels).sum().item()
        print('Accuracy:', 100*(correct/total),'%')